# Laboratorio 3 — La tarea de clasificación
## Caso: Ruta Alpes — Recomendación de cupones (Cafetería y Restaurante(<20))

**Curso:** ISIS-2611 Aprendizaje de Máquina — Universidad de los Andes

**Integrantes:**
- Julian Pinto
- _Integrante 2_

---

**Objetivo del negocio.** Ruta Alpes quiere saber, para un tipo de cupón dado, *cuándo* un conductor tiene mayor probabilidad de aceptarlo, para dejar de enviar la misma oferta a todos los usuarios. Se construyen clasificadores binarios (`acepta_cupon` = Sí / No) para dos tipos de cupón: **Cafetería** y **Restaurante(<20)**. Se comparan regresión logística, árboles de decisión y Random Forest; se evalúa el balanceo de clases en Restaurante(<20) y se discuten riesgos de sesgo del uso de variables demográficas.

**Contenido**
1. Exploración y calidad de datos
2. Limpieza y preparación
3. Regresión logística
4. Árboles de decisión
5. Random Forest
6. Balanceo de clases (SMOTENC) en Restaurante(<20)
7. Tabla comparativa y selección del mejor modelo
8. Variables más relevantes
9. Predicciones sobre `Lab3_datos_Prueba`
10. Análisis de resultados
11. Uso de herramientas de IA generativa

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay,
                             precision_score, recall_score, f1_score)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTENC

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42
CUPONES = ['Cafetería', 'Restaurante(<20)']

# 1. Exploración de los datos y análisis de calidad

Se carga el conjunto completo y el diccionario de datos. La exploración se hace **por subconjunto** (Cafetería y Restaurante(<20)), porque cada uno será un problema de clasificación independiente.

In [ ]:
datos_raw = pd.read_excel('Data/Lab3_Datos.xlsx')
diccionario = pd.read_excel('Data/Lab3_diccionario_Datos.xlsx')
data = datos_raw.copy()
print('Dimensiones del conjunto completo:', data.shape)
data.head()

In [ ]:
diccionario

In [ ]:
data['cupon'].value_counts().to_frame('registros')

Se separan los dos subconjuntos de interés. Los demás tipos de cupón (Bar, Comida para llevar, Restaurante(20-50)) no se utilizan en este laboratorio.

In [ ]:
subconjuntos = {c: data[data['cupon'] == c].reset_index(drop=True) for c in CUPONES}
for c, df in subconjuntos.items():
    print(f'{c:<18} -> {df.shape[0]} filas, {df.shape[1]} columnas')

In [ ]:
subconjuntos['Cafetería'].describe(include='all').T

## 1.1 Distribución de la variable objetivo `acepta_cupon`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
dist_obj = {}
for ax, (c, df) in zip(axes, subconjuntos.items()):
    conteo = df['acepta_cupon'].value_counts().reindex(['No', 'Sí'])
    dist_obj[c] = (conteo / conteo.sum() * 100).round(1)
    ax.bar(conteo.index, conteo.values, color=['#FF6B6B', '#4C6EF5'])
    for i, v in enumerate(conteo.values):
        ax.text(i, v + 20, f'{v} ({v / conteo.sum() * 100:.1f}%)', ha='center')
    ax.set(title=f'acepta_cupon — {c}', ylabel='Número de registros', ylim=(0, conteo.max() * 1.15))
plt.show()

pd.DataFrame(dist_obj).T.rename(columns=lambda x: f'% {x}')

**Hallazgo.**
- **Cafetería** está prácticamente **balanceado** (≈50/50).
- **Restaurante(<20)** está **desbalanceado** (≈71% *Sí* / 29% *No*). La **clase minoritaria es “No” (rechaza el cupón)**. Esto es importante: un modelo que prediga casi siempre *Sí* tendrá una exactitud alta pero no servirá para identificar a quién *no* conviene enviarle el cupón (el caso de notificaciones molestas). Por eso se usará F1-macro como métrica de selección y se analizará el recall de la clase *No* (punto 6).

## 1.2 Completitud (valores faltantes)

In [ ]:
nulos = pd.DataFrame({c: df.isna().mean() * 100 for c, df in subconjuntos.items()}).round(2)
nulos = nulos[(nulos > 0).any(axis=1)]
display(nulos.rename(columns=lambda x: f'% nulos {x}'))

fig, ax = plt.subplots(figsize=(10, 3.5), constrained_layout=True)
nulos.drop(index='vehiculo').plot(kind='bar', ax=ax, color=['#4C6EF5', '#FF6B6B'])
ax.set(ylabel='% de nulos', title='Porcentaje de nulos en variables de frecuencia (sin vehiculo, que tiene ~99%)')
plt.xticks(rotation=20, ha='right')
plt.show()

**Hallazgo.**
- `vehiculo` tiene **~99% de valores faltantes** en ambos subconjuntos (y 100% en el archivo de prueba). No es recuperable.
- Las variables `frecuencia_*` tienen entre **1% y 2%** de faltantes: un porcentaje bajo que se puede imputar.
- El resto de variables está completo.

## 1.3 Unicidad (registros duplicados)

In [ ]:
pd.DataFrame({
    'duplicados': {c: df.duplicated().sum() for c, df in subconjuntos.items()},
    '% del subconjunto': {c: round(df.duplicated().mean() * 100, 2) for c, df in subconjuntos.items()},
})

**Hallazgo.** Hay 7 filas exactamente duplicadas en cada subconjunto (<0.3%). No existe un identificador de conductor, por lo que no se puede confirmar si son el mismo registro; sin embargo, dejarlas permitiría que una copia quede en *train* y otra en *test*, inflando las métricas.

## 1.4 Validez (valores dentro del dominio del diccionario)

Se compara el conjunto de valores observados de cada variable contra los valores permitidos según el diccionario.

In [ ]:
FREC = ['nunca', 'menos de 1', '1 a 3', '4 a 8', 'más de 8']
DOMINIOS = {
    'destino': ['Casa', 'Sin lugar urgente', 'Trabajo'],
    'acompanante': ['Solo', 'Amigo(s)', 'Hijo(s)', 'Pareja'],
    'clima': ['Soleado', 'Lluvioso', 'Nevado'],
    'temperatura': [30, 55, 80],
    'hora': ['7:00 a.m.', '10:00 a.m.', '2:00 p.m.', '6:00 p.m.', '10:00 p.m.'],
    'vigencia_cupon': ['2 horas', '1 día'],
    'genero': ['Femenino', 'Masculino'],
    'edad': ['menor de 21', '21', '26', '31', '36', '41', '46', '50 o más'],
    'estado_civil': ['Soltero(a)', 'Casado(a)', 'Unión libre', 'Divorciado(a)', 'Viudo(a)'],
    'tiene_hijos': ['Sí', 'No'],
    'educacion': ['Bachillerato incompleto', 'Bachillerato completo', 'Estudios universitarios sin título',
                  'Técnico/Tecnólogo', 'Pregrado', 'Posgrado (Maestría o Doctorado)'],
    'ingreso': ['Menos de $12.500', '$12.500 - $24.999', '$25.000 - $37.499', '$37.500 - $49.999',
                '$50.000 - $62.499', '$62.500 - $74.999', '$75.000 - $87.499', '$87.500 - $99.999',
                '$100.000 o más'],
    'vehiculo': ['Mazda5', 'Scooter o motocicleta', 'Camioneta tipo crossover',
                 'Carro muy viejo para instalarle Onstar', 'No conduce'],
    'frecuencia_bar': FREC, 'frecuencia_cafeteria': FREC, 'frecuencia_comida_para_llevar': FREC,
    'frecuencia_restaurante_menos_20': FREC, 'frecuencia_restaurante_20_a_50': FREC,
    'tiempo_cupon_mayor_5min': ['Sí', 'No'], 'tiempo_cupon_mayor_15min': ['Sí', 'No'],
    'tiempo_cupon_mayor_25min': ['Sí', 'No'], 'misma_direccion': ['Sí', 'No'],
    'direccion_opuesta': ['Sí', 'No'], 'acepta_cupon': ['Sí', 'No'],
}

filas = []
for c, df in subconjuntos.items():
    for col, dominio in DOMINIOS.items():
        fuera = set(df[col].dropna().unique()) - set(dominio)
        filas.append({'subconjunto': c, 'variable': col, 'n únicos': df[col].nunique(),
                      'valores fuera del dominio': ', '.join(map(str, fuera)) if fuera else '—'})
validez = pd.DataFrame(filas)
validez[validez['valores fuera del dominio'] != '—']

In [ ]:
print('Variables con un único valor (constantes):')
for c, df in subconjuntos.items():
    print(f'  {c}:', [col for col in df.columns if df[col].nunique() == 1])
print('\nNúmero de ocupaciones distintas:', data['ocupacion'].nunique())

**Hallazgo.**
- Todas las variables están dentro del dominio esperado, salvo una diferencia de texto en `vehiculo` (`"... Onstar :D"`), que de todas formas es una variable casi vacía.
- `tiempo_cupon_mayor_5min` es **constante** ("Sí" en todos los registros): no aporta información. `cupon` también es constante dentro de cada subconjunto, por construcción.
- `ocupacion` tiene 25 categorías, algunas con muy pocos registros (ver 1.6).

## 1.5 Consistencia entre variables

In [ ]:
df_c = subconjuntos['Cafetería']
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), constrained_layout=True)
sns.heatmap(pd.crosstab(data['clima'], data['temperatura']), annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0])
axes[0].set_title('clima vs temperatura')
sns.heatmap(pd.crosstab(data['tiempo_cupon_mayor_15min'], data['tiempo_cupon_mayor_25min']), annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[1])
axes[1].set_title('tiempo >15 min vs tiempo >25 min')
sns.heatmap(pd.crosstab(data['misma_direccion'], data['direccion_opuesta']), annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[2])
axes[2].set_title('misma_direccion vs direccion_opuesta')
plt.show()

**Hallazgo.**
- `clima` y `temperatura` son **lógicamente consistentes** (no hay "Nevado" a 80°F), pero `temperatura` está casi determinada por `clima`: Lluvioso → 55°F, Nevado → 30°F. Solo aporta información adicional dentro de "Soleado". El diccionario aclara además que es una *variable de diseño* de la encuesta con solo 3 valores.
- `tiempo_cupon_mayor_15min` y `tiempo_cupon_mayor_25min` son consistentes (no existe >25 min sin >15 min). Juntas codifican tres niveles de distancia.
- `direccion_opuesta` es **exactamente el complemento** de `misma_direccion`: redundancia perfecta.

Para cuantificar la redundancia entre todas las variables se calcula la **V de Cramér** (asociación entre variables categóricas, de 0 a 1).

In [ ]:
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    tabla = pd.crosstab(x, y)
    if min(tabla.shape) < 2:
        return np.nan
    chi2 = chi2_contingency(tabla, correction=False)[0]
    n = tabla.values.sum()
    return np.sqrt(chi2 / (n * (min(tabla.shape) - 1)))

cols_v = [c for c in df_c.columns if c not in ['cupon', 'vehiculo', 'tiempo_cupon_mayor_5min']]
matriz_v = pd.DataFrame(index=cols_v, columns=cols_v, dtype=float)
for a in cols_v:
    for b in cols_v:
        matriz_v.loc[a, b] = cramers_v(df_c[a], df_c[b])

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(matriz_v, cmap='Blues', vmin=0, vmax=1, ax=ax, annot=True, fmt='.1f', annot_kws={'size': 7})
ax.set_title('V de Cramér entre variables — subconjunto Cafetería')
plt.show()

pares = (matriz_v.where(np.triu(np.ones(matriz_v.shape, dtype=bool), k=1)).stack()
         .sort_values(ascending=False).head(10).round(3).to_frame('V de Cramér'))
pares

**Hallazgo.** Las asociaciones más altas confirman las redundancias: `misma_direccion`–`direccion_opuesta` (V = 1), `clima`–`temperatura`, y las asociaciones entre variables del perfil (por ejemplo, `ocupacion`–`edad` y `estado_civil`–`tiene_hijos`). Las del perfil son esperables y no se eliminan, porque cada una aporta información distinta.

## 1.6 Valores atípicos y categorías poco frecuentes

La única variable numérica es `temperatura`, que solo toma 3 valores fijos ({30, 55, 80}); no tiene atípicos en el sentido clásico (IQR). En variables categóricas, el equivalente son las **categorías con muy pocos registros**, que pueden generar coeficientes inestables o reglas poco confiables.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
for ax, (c, df) in zip(axes, subconjuntos.items()):
    vc = df['ocupacion'].value_counts(normalize=True).sort_values() * 100
    ax.barh(vc.index, vc.values, color=np.where(vc.values < 1, '#FF6B6B', '#4C6EF5'))
    ax.axvline(1, color='gray', ls='--', lw=1)
    ax.set(title=f'ocupacion — {c} (rojo: <1%)', xlabel='% de registros')
plt.show()

raras = {c: {col: (df[col].value_counts(normalize=True) < 0.01).sum()
             for col in df.columns if df[col].dtype == object and df[col].nunique() > 2}
         for c, df in subconjuntos.items()}
raras = pd.DataFrame(raras)
raras[(raras > 0).any(axis=1)].rename(columns=lambda x: f'# categorías con <1% ({x})')

**Hallazgo.** Hay categorías poco frecuentes (algunas ocupaciones, `Viudo(a)`, `Bachillerato incompleto`), pero son valores **válidos** del dominio y no errores. No se eliminan: el One-Hot con `handle_unknown='ignore'` y la regularización (en logística) o `min_samples_leaf` (en árboles) mitigan su efecto.

## 1.7 Relación de las variables con la aceptación del cupón

Tasa de aceptación (% de *Sí*) por categoría, comparando ambos tipos de cupón. La línea punteada marca la tasa global de cada subconjunto.

In [ ]:
vars_plot = ['destino', 'acompanante', 'clima', 'hora', 'vigencia_cupon', 'tiempo_cupon_mayor_15min',
             'tiempo_cupon_mayor_25min', 'misma_direccion', 'frecuencia_cafeteria',
             'frecuencia_restaurante_menos_20', 'edad', 'ingreso', 'genero', 'estado_civil', 'tiene_hijos', 'educacion']
ambos = pd.concat(subconjuntos.values())
ambos['acepta'] = (ambos['acepta_cupon'] == 'Sí').astype(int)

fig, axes = plt.subplots(4, 4, figsize=(22, 20), constrained_layout=True)
for ax, var in zip(axes.flat, vars_plot):
    orden = [v for v in DOMINIOS[var] if v in ambos[var].unique()]
    tasa = ambos.groupby(['cupon', var])['acepta'].mean().mul(100).reset_index()
    sns.barplot(data=tasa, x=var, y='acepta', hue='cupon', order=orden, hue_order=CUPONES,
                palette=['#4C6EF5', '#FF6B6B'], ax=ax)
    for color, c in zip(['#4C6EF5', '#FF6B6B'], CUPONES):
        ax.axhline(ambos.loc[ambos['cupon'] == c, 'acepta'].mean() * 100, color=color, ls='--', lw=1)
    ax.set(title=var, xlabel='', ylabel='% acepta', ylim=(0, 100))
    ax.tick_params(axis='x', rotation=35)
    for lbl in ax.get_xticklabels():
        lbl.set_ha('right')
    ax.legend(fontsize=8, loc='lower right')
plt.show()

**Hallazgos preliminares (a confirmar con los modelos):**
- **Contexto del trayecto:** `destino` "Sin lugar urgente", ir con amigos/pareja, clima soleado, que el lugar quede a menos de 25 minutos y en la misma dirección aumentan la aceptación en ambos cupones.
- **Cafetería:** la `hora` y la `vigencia_cupon` tienen un efecto marcado (la aceptación cae en la noche y con vigencia de 2 horas), y sobre todo la **frecuencia con que el conductor visita cafeterías**: quienes nunca van casi no aceptan.
- **Restaurante(<20):** la aceptación es alta en casi todas las categorías (la clase base ya es 71%). Las diferencias son más suaves, lo que anticipa un problema más difícil de discriminar.
- **Demografía:** `edad`, `ingreso` y `educacion` muestran diferencias moderadas. Este punto se retoma en el análisis de sesgo.

# 2. Limpieza y preparación de los datos

Decisiones tomadas por el equipo, a partir de los hallazgos del paso 1:

| # | Decisión | Justificación (hallazgo) |
|---|---|---|
| 1 | **Eliminar `vehiculo`** | ~99% nulos en entrenamiento y 100% en prueba (1.2). No es imputable de forma confiable. |
| 2 | **Eliminar `tiempo_cupon_mayor_5min`** | Constante (1.4): varianza cero, no aporta información. |
| 3 | **Eliminar `direccion_opuesta`** | Complemento exacto de `misma_direccion` (1.5, V de Cramér = 1). Redundancia perfecta. |
| 4 | **Eliminar `temperatura`** | Variable de diseño de la encuesta, casi determinada por `clima` (1.5). Se conserva `clima`, que es más interpretable. |
| 5 | **Eliminar `cupon`** | Constante dentro de cada subconjunto (cada modelo es de un solo tipo de cupón). |
| 6 | **Eliminar duplicados exactos** | 7 por subconjunto (1.3). Evita que copias queden en train y test simultáneamente. |
| 7 | **Imputar con la moda (`most_frequent`) dentro del pipeline** las `frecuencia_*` | 1–2% de faltantes (1.2). Imputar dentro del pipeline evita fuga de información (la moda se aprende solo con el fold de entrenamiento) y permite procesar datos nuevos con faltantes. |
| 8 | **`OrdinalEncoder` con el orden del diccionario** para `edad`, `educacion`, `ingreso`, `hora` y las 5 `frecuencia_*` | Son ordinales (diccionario): el orden tiene significado y se reduce la dimensionalidad frente a One-Hot. |
| 9 | **`OneHotEncoder(drop='if_binary', handle_unknown='ignore')`** para las nominales | `destino`, `acompanante`, `clima`, `estado_civil`, `ocupacion` no tienen orden. Las binarias quedan en una sola columna. `handle_unknown='ignore'` protege ante categorías nuevas en prueba. |
| 10 | **`StandardScaler` solo para regresión logística** | La regularización L1/L2 penaliza coeficientes y es sensible a la escala. Los árboles y Random Forest son invariantes a la escala. |
| 11 | **Partición 80/20 estratificada** y **`StratifiedKFold(5)`** | La estratificación conserva la proporción de clases (clave en Restaurante(<20)). 5 folds es un buen compromiso entre estabilidad y costo de cómputo. |
| 12 | **Métrica de selección: F1-macro** | Promedia el F1 de *Sí* y *No*. No premia predecir siempre la clase mayoritaria en Restaurante(<20). |
| 13 | **Variable objetivo: `Sí` → 1, `No` → 0** | La clase positiva (1) es *acepta el cupón*. |

In [ ]:
COLS_ELIMINAR = ['vehiculo', 'tiempo_cupon_mayor_5min', 'direccion_opuesta', 'temperatura', 'cupon']

COLS_ORDINALES = ['edad', 'educacion', 'ingreso', 'hora', 'frecuencia_bar', 'frecuencia_cafeteria',
                  'frecuencia_comida_para_llevar', 'frecuencia_restaurante_menos_20', 'frecuencia_restaurante_20_a_50']
COLS_NOMINALES = ['destino', 'acompanante', 'clima', 'vigencia_cupon', 'genero', 'estado_civil', 'tiene_hijos',
                  'ocupacion', 'tiempo_cupon_mayor_15min', 'tiempo_cupon_mayor_25min', 'misma_direccion']
ORDEN_ORDINALES = [DOMINIOS[c] for c in COLS_ORDINALES]


def preparar(df):
    # Elimina duplicados exactos (sobre el registro original) y las columnas descartadas
    df = df.drop_duplicates().reset_index(drop=True)
    X = df.drop(columns=COLS_ELIMINAR + ['acepta_cupon'])
    y = (df['acepta_cupon'] == 'Sí').astype(int)
    return X, y


splits = {}
for c, df in subconjuntos.items():
    X, y = preparar(df)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
    splits[c] = (X_train, X_test, y_train, y_test)
    print(f'{c:<18} train: {X_train.shape}  test: {X_test.shape}  '
          f'% Sí train: {y_train.mean() * 100:.1f}  % Sí test: {y_test.mean() * 100:.1f}')

assert set(COLS_ORDINALES + COLS_NOMINALES) == set(X.columns)

### Pipeline de preprocesamiento

Todos los modelos comparten la misma estructura de pipeline (de `imblearn`, que es compatible con `sklearn` y permite insertar el paso de balanceo en el punto 6):

```
imputador (moda) → codificación ordinal → [SMOTENC, solo en el punto 6] → escalado (solo logística) + One-Hot nominales → modelo
```

La codificación ordinal se hace **antes** del balanceo para que SMOTENC trate las variables ordinales como numéricas (interpolando entre niveles vecinos) y las nominales como categóricas.

In [ ]:
def crear_pipeline(modelo, escalar=False, balanceo=False):
    imputador = SimpleImputer(strategy='most_frequent').set_output(transform='pandas')

    ordinal = ColumnTransformer(
        [('ord', OrdinalEncoder(categories=ORDEN_ORDINALES), COLS_ORDINALES)],
        remainder='passthrough', verbose_feature_names_out=False,
    ).set_output(transform='pandas')

    preprocesador = ColumnTransformer([
        ('ord', StandardScaler() if escalar else 'passthrough', COLS_ORDINALES),
        ('nom', OneHotEncoder(drop='if_binary', handle_unknown='ignore', sparse_output=False), COLS_NOMINALES),
    ])

    pasos = [('imputador', imputador), ('ordinal', ordinal)]
    if balanceo:
        pasos.append(('balanceo', SMOTENC(categorical_features=COLS_NOMINALES, random_state=RANDOM_STATE)))
    pasos += [('preprocesador', preprocesador), ('modelo', modelo)]
    return ImbPipeline(pasos)


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
crear_pipeline(LogisticRegression(), escalar=True)

In [ ]:
def metricas(y_true, y_pred):
    return {
        'Precisión (Sí)': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'Recall (Sí)': recall_score(y_true, y_pred, pos_label=1),
        'F1 (Sí)': f1_score(y_true, y_pred, pos_label=1),
        'Precisión (No)': precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        'Recall (No)': recall_score(y_true, y_pred, pos_label=0),
        'F1 (No)': f1_score(y_true, y_pred, pos_label=0),
        'F1-macro': f1_score(y_true, y_pred, average='macro'),
    }


def buscar(pipeline, param_grid, cupon):
    X_train, X_test, y_train, y_test = splits[cupon]
    grid = GridSearchCV(pipeline, param_grid, cv=cv, scoring='f1_macro', n_jobs=-1, return_train_score=True)
    grid.fit(X_train, y_train)
    return grid


def resumen_grid(grid, cols_param, top=8):
    res = pd.DataFrame(grid.cv_results_)
    cols = [f'param_{p}' for p in cols_param] + ['mean_train_score', 'mean_test_score', 'std_test_score', 'rank_test_score']
    return (res[cols].sort_values('rank_test_score').head(top)
            .rename(columns=lambda x: x.replace('param_modelo__', '')).round(4))


def evaluar(grid, cupon, nombre):
    X_train, X_test, y_train, y_test = splits[cupon]
    modelo = grid.best_estimator_
    y_pred_tr, y_pred_te = modelo.predict(X_train), modelo.predict(X_test)
    print(f'=== {nombre} — {cupon} ===')
    print('Mejores hiperparámetros:', grid.best_params_)
    print(f'F1-macro CV: {grid.best_score_:.4f}')
    tabla = pd.DataFrame({'Train': metricas(y_train, y_pred_tr), 'Test': metricas(y_test, y_pred_te)}).T.round(4)
    display(tabla)
    print(classification_report(y_test, y_pred_te, target_names=['No', 'Sí']))
    return metricas(y_test, y_pred_te)


def matrices(grids, nombre):
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)
    for ax, c in zip(axes, CUPONES):
        X_train, X_test, y_train, y_test = splits[c]
        ConfusionMatrixDisplay.from_predictions(y_test, grids[c].predict(X_test), display_labels=['No', 'Sí'],
                                                cmap='Blues', ax=ax, colorbar=False)
        ax.set_title(f'{nombre} — {c}')
    plt.show()


resultados_test = {}  # (cupón, modelo) -> métricas en test

# 3. Regresión logística

Pipeline: imputación → codificación ordinal → `StandardScaler` (ordinales) + One-Hot (nominales) → `LogisticRegression`.

**Búsqueda de hiperparámetros** (`GridSearchCV`, 5 folds estratificados, F1-macro):
- **Tipo de regularización**: L2, L1 y ElasticNet (mezcla 50/50). En scikit-learn 1.9 el parámetro `penalty` está obsoleto y la regularización se controla con `l1_ratio` (0 = L2, 1 = L1, 0.5 = ElasticNet).
- **Solver**: `lbfgs`, `newton-cg`, `liblinear` y `saga`. Solo se prueban las combinaciones válidas: L1 únicamente con `liblinear`/`saga` y ElasticNet únicamente con `saga`.
- **C** (inverso de la fuerza de regularización): 0.01, 0.1, 1 y 10.

In [ ]:
C_VALORES = [0.01, 0.1, 1, 10]
param_grid_rl = [
    {'modelo__l1_ratio': [0.0], 'modelo__solver': ['lbfgs', 'newton-cg', 'liblinear', 'saga'], 'modelo__C': C_VALORES},
    {'modelo__l1_ratio': [1.0], 'modelo__solver': ['liblinear', 'saga'], 'modelo__C': C_VALORES},
    {'modelo__l1_ratio': [0.5], 'modelo__solver': ['saga'], 'modelo__C': C_VALORES},
]

grids_rl = {}
for c in CUPONES:
    pipe = crear_pipeline(LogisticRegression(max_iter=5000, random_state=RANDOM_STATE), escalar=True)
    grids_rl[c] = buscar(pipe, param_grid_rl, c)

In [ ]:
for c in CUPONES:
    display(Markdown(f'**Top combinaciones — Regresión logística — {c}**'))
    display(resumen_grid(grids_rl[c], ['modelo__l1_ratio', 'modelo__solver', 'modelo__C']))

In [ ]:
for c in CUPONES:
    resultados_test[(c, 'Regresión logística')] = evaluar(grids_rl[c], c, 'Regresión logística')
matrices(grids_rl, 'Reg. logística')

**Análisis — Regresión logística.**

_[COMPLETAR tras revisar los resultados: hiperparámetros elegidos y qué significan, sobreajuste train vs test, diferencias entre cupones.]_

# 4. Árboles de decisión

Pipeline: imputación → codificación ordinal → One-Hot (nominales) → `DecisionTreeClassifier`. No se escala porque los árboles son invariantes a transformaciones monótonas.

**Búsqueda de hiperparámetros:**
- `max_depth`: 3, 5, 7, 10, 15, None. Controla la complejidad: un árbol muy profundo memoriza el entrenamiento.
- `min_samples_leaf`: 1, 5, 10, 20, 50. Hojas más grandes producen reglas más generales y estables.
- `criterion`: gini, entropy, log_loss. Medida de impureza usada para elegir las divisiones.

In [ ]:
param_grid_dt = {
    'modelo__max_depth': [3, 5, 7, 10, 15, None],
    'modelo__min_samples_leaf': [1, 5, 10, 20, 50],
    'modelo__criterion': ['gini', 'entropy', 'log_loss'],
}

grids_dt = {}
for c in CUPONES:
    pipe = crear_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE))
    grids_dt[c] = buscar(pipe, param_grid_dt, c)

for c in CUPONES:
    display(Markdown(f'**Top combinaciones — Árbol de decisión — {c}**'))
    display(resumen_grid(grids_dt[c], ['modelo__max_depth', 'modelo__min_samples_leaf', 'modelo__criterion']))

In [ ]:
for c in CUPONES:
    resultados_test[(c, 'Árbol de decisión')] = evaluar(grids_dt[c], c, 'Árbol de decisión')
matrices(grids_dt, 'Árbol')

In [ ]:
# Efecto de la profundidad sobre el F1-macro (train vs validación), con los demás parámetros en su mejor valor
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), constrained_layout=True)
for ax, c in zip(axes, CUPONES):
    res = pd.DataFrame(grids_dt[c].cv_results_)
    bp = grids_dt[c].best_params_
    sel = res[(res['param_modelo__min_samples_leaf'] == bp['modelo__min_samples_leaf'])
              & (res['param_modelo__criterion'] == bp['modelo__criterion'])].copy()
    sel['depth'] = sel['param_modelo__max_depth'].astype(str)
    ax.plot(sel['depth'], sel['mean_train_score'], 'o-', label='Train (CV)', color='#4C6EF5')
    ax.plot(sel['depth'], sel['mean_test_score'], 'o-', label='Validación (CV)', color='#FF6B6B')
    ax.set(title=f'{c} — min_samples_leaf={bp["modelo__min_samples_leaf"]}, {bp["modelo__criterion"]}',
           xlabel='max_depth', ylabel='F1-macro')
    ax.legend()
plt.show()

**Análisis — Árboles de decisión.**

_[COMPLETAR tras revisar los resultados.]_

# 5. Random Forest

Pipeline: imputación → codificación ordinal → One-Hot → `RandomForestClassifier` (`max_features='sqrt'`, el valor por defecto).

**Búsqueda de hiperparámetros:**
- `n_estimators`: 100, 200, 400. Más árboles reducen la varianza del ensamble, con rendimientos decrecientes.
- `max_depth`: 5, 10, 15, 20, None.
- `min_samples_leaf`: 1, 2, 5, 10.

In [ ]:
param_grid_rf = {
    'modelo__n_estimators': [100, 200, 400],
    'modelo__max_depth': [5, 10, 15, 20, None],
    'modelo__min_samples_leaf': [1, 2, 5, 10],
}

grids_rf = {}
for c in CUPONES:
    pipe = crear_pipeline(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1))
    grids_rf[c] = buscar(pipe, param_grid_rf, c)

for c in CUPONES:
    display(Markdown(f'**Top combinaciones — Random Forest — {c}**'))
    display(resumen_grid(grids_rf[c], ['modelo__n_estimators', 'modelo__max_depth', 'modelo__min_samples_leaf']))

In [ ]:
for c in CUPONES:
    resultados_test[(c, 'Random Forest')] = evaluar(grids_rf[c], c, 'Random Forest')
matrices(grids_rf, 'Random Forest')

**Análisis — Random Forest.**

_[COMPLETAR tras revisar los resultados.]_

# 6. Balanceo de clases en Restaurante(<20) con SMOTENC

En Restaurante(<20) la clase minoritaria es **No** (≈29%). Se evalúa si sobremuestrearla mejora los modelos.

**¿Por qué SMOTENC y no SMOTE?** SMOTE interpola entre vecinos en un espacio numérico. Con variables One-Hot o nominales generaría registros imposibles (por ejemplo, 0.6 de "Casa" y 0.4 de "Trabajo"). SMOTENC (*SMOTE for Nominal and Continuous*) interpola solo las variables numéricas (aquí, las ordinales codificadas) y asigna a las nominales la categoría más frecuente entre los vecinos.

**Cómo se evita la fuga de datos:** el paso `balanceo` está dentro del pipeline de `imblearn`, por lo que en cada fold de la validación cruzada solo se sobremuestrea el fold de entrenamiento. La validación y el test conservan la distribución real.

Se aplica a los **tres algoritmos**, con los mismos grids de hiperparámetros, para comparar en igualdad de condiciones.

In [ ]:
CUPON_DESB = 'Restaurante(<20)'

# Verificación: distribución antes y después de SMOTENC sobre el train completo
_pipe = crear_pipeline(LogisticRegression(), balanceo=True)
_Xo = _pipe[:2].fit_transform(splits[CUPON_DESB][0])
_Xb, _yb = _pipe.named_steps['balanceo'].fit_resample(_Xo, splits[CUPON_DESB][2])
pd.DataFrame({'Antes': splits[CUPON_DESB][2].value_counts(), 'Después de SMOTENC': _yb.value_counts()}).rename(index={0: 'No', 1: 'Sí'})

In [ ]:
grids_bal = {}
config = {
    'Regresión logística': (LogisticRegression(max_iter=5000, random_state=RANDOM_STATE), True, param_grid_rl),
    'Árbol de decisión': (DecisionTreeClassifier(random_state=RANDOM_STATE), False, param_grid_dt),
    'Random Forest': (RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1), False, param_grid_rf),
}
for nombre, (modelo, escalar, grid) in config.items():
    grids_bal[nombre] = buscar(crear_pipeline(modelo, escalar=escalar, balanceo=True), grid, CUPON_DESB)
    print(f'{nombre:<20} mejores: {grids_bal[nombre].best_params_}  F1-macro CV: {grids_bal[nombre].best_score_:.4f}')

In [ ]:
grids_sin = {'Regresión logística': grids_rl[CUPON_DESB], 'Árbol de decisión': grids_dt[CUPON_DESB], 'Random Forest': grids_rf[CUPON_DESB]}
X_train, X_test, y_train, y_test = splits[CUPON_DESB]

filas = {}
for nombre in config:
    for etiqueta, g in [('Sin balanceo', grids_sin[nombre]), ('Con SMOTENC', grids_bal[nombre])]:
        m = metricas(y_test, g.predict(X_test))
        m['F1-macro CV'] = g.best_score_
        filas[(nombre, etiqueta)] = m
comp_bal = pd.DataFrame(filas).T[['F1-macro CV', 'F1-macro', 'F1 (Sí)', 'Recall (No)', 'Precisión (No)', 'F1 (No)', 'Recall (Sí)', 'Precisión (Sí)']].round(4)
comp_bal

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True, sharey=True)
for ax, nombre in zip(axes, config):
    comp_bal.loc[nombre, ['F1-macro', 'F1 (Sí)', 'Recall (No)', 'Precisión (No)']].T.plot(
        kind='bar', ax=ax, color=['#4C6EF5', '#FF6B6B'])
    ax.set(title=f'{nombre} — {CUPON_DESB}', ylim=(0, 1), ylabel='Valor en test')
    ax.tick_params(axis='x', rotation=0)
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(13, 7), constrained_layout=True)
for j, nombre in enumerate(config):
    for i, (etiqueta, g) in enumerate([('Sin balanceo', grids_sin[nombre]), ('Con SMOTENC', grids_bal[nombre])]):
        ConfusionMatrixDisplay.from_predictions(y_test, g.predict(X_test), display_labels=['No', 'Sí'],
                                                cmap='Blues' if i == 0 else 'Oranges', ax=axes[i, j], colorbar=False)
        axes[i, j].set_title(f'{nombre}\n{etiqueta}')
plt.show()

**Análisis y recomendación sobre el balanceo.**

_[COMPLETAR — DECISIÓN DEL EQUIPO.]_

# 7. Tabla comparativa de modelos (conjunto de test)

Métricas sobre el **conjunto de test** (20% no visto durante la búsqueda de hiperparámetros). Se reportan precisión, recall y F1 de la clase positiva (*Sí, acepta*), el recall de la clase *No* y el F1-macro (métrica de selección).

In [ ]:
for nombre in config:
    m = metricas(y_test, grids_bal[nombre].predict(X_test))
    resultados_test[(CUPON_DESB, f'{nombre} + SMOTENC')] = m

tabla_comp = pd.DataFrame(resultados_test).T
tabla_comp.index.names = ['Cupón', 'Modelo']
tabla_comp = tabla_comp[['Precisión (Sí)', 'Recall (Sí)', 'F1 (Sí)', 'Recall (No)', 'F1 (No)', 'F1-macro']].sort_index(level=0, sort_remaining=False)

cv_scores = {(c, 'Regresión logística'): grids_rl[c].best_score_ for c in CUPONES}
cv_scores |= {(c, 'Árbol de decisión'): grids_dt[c].best_score_ for c in CUPONES}
cv_scores |= {(c, 'Random Forest'): grids_rf[c].best_score_ for c in CUPONES}
cv_scores |= {(CUPON_DESB, f'{n} + SMOTENC'): grids_bal[n].best_score_ for n in config}
tabla_comp.insert(0, 'F1-macro CV', pd.Series(cv_scores))
tabla_comp.round(4).style.highlight_max(subset=pd.IndexSlice[:, ['F1-macro', 'F1 (Sí)']], color='#d0ebff', axis=0).format('{:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5), constrained_layout=True)
for ax, c in zip(axes, CUPONES):
    tabla_comp.loc[c, ['Precisión (Sí)', 'Recall (Sí)', 'F1 (Sí)', 'F1-macro']].plot(kind='bar', ax=ax,
        color=['#4C6EF5', '#FF6B6B', '#51CF66', '#868E96'])
    ax.set(title=f'Desempeño en test — {c}', ylim=(0, 1), ylabel='Valor', xlabel='')
    ax.tick_params(axis='x', rotation=15)
    ax.legend(loc='lower right', fontsize=8)
plt.show()

**Selección del mejor modelo por cupón.**

_[COMPLETAR — DECISIÓN DEL EQUIPO.]_

# 8. Variables más relevantes

Se usan tres perspectivas complementarias:
1. **Importancia por permutación** del Random Forest sobre el test: cuánto cae el F1-macro al desordenar aleatoriamente cada variable *original*. Es independiente de la codificación y se mide en datos no vistos.
2. **Coeficientes de la regresión logística**: dan la dirección del efecto (+ aumenta la probabilidad de aceptar, − la disminuye). Son comparables entre sí porque las ordinales están estandarizadas.
3. **Reglas del árbol de decisión** (en la sección de análisis).

In [ ]:
importancias = {}
for c in CUPONES:
    X_train, X_test, y_train, y_test = splits[c]
    r = permutation_importance(grids_rf[c].best_estimator_, X_test, y_test, scoring='f1_macro',
                               n_repeats=15, random_state=RANDOM_STATE, n_jobs=-1)
    importancias[c] = pd.DataFrame({'media': r.importances_mean, 'std': r.importances_std}, index=X_test.columns)

fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)
for ax, c in zip(axes, CUPONES):
    imp = importancias[c].sort_values('media')
    ax.barh(imp.index, imp['media'], xerr=imp['std'], color='#4C6EF5')
    ax.axvline(0, color='gray', lw=1)
    ax.set(title=f'Importancia por permutación (RF) — {c}', xlabel='Caída en F1-macro al permutar')
plt.show()

pd.concat({c: importancias[c]['media'].rank(ascending=False).astype(int) for c in CUPONES}, axis=1) \
  .assign(**{f'Imp. {c}': importancias[c]['media'].round(4) for c in CUPONES}) \
  .sort_values('Cafetería')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8), constrained_layout=True)
coefs_rl = {}
for ax, c in zip(axes, CUPONES):
    best = grids_rl[c].best_estimator_
    nombres = best.named_steps['preprocesador'].get_feature_names_out()
    nombres = [n.split('__', 1)[1] for n in nombres]
    coef = pd.Series(best.named_steps['modelo'].coef_[0], index=nombres)
    coefs_rl[c] = coef
    top = pd.concat([coef.nlargest(10), coef.nsmallest(10)]).sort_values()
    ax.barh(top.index, top.values, color=np.where(top.values > 0, '#4C6EF5', '#FF6B6B'))
    ax.axvline(0, color='gray', lw=1)
    ax.set(title=f'Coeficientes regresión logística (top ±10) — {c}', xlabel='Coeficiente (log-odds)')
plt.show()

In [ ]:
# Reglas del árbol de decisión (primeros 3 niveles) para cada cupón
for c in CUPONES:
    best = grids_dt[c].best_estimator_
    nombres = [n.split('__', 1)[1] for n in best.named_steps['preprocesador'].get_feature_names_out()]
    fig, ax = plt.subplots(figsize=(22, 8))
    plot_tree(best.named_steps['modelo'], max_depth=3, feature_names=nombres, class_names=['No', 'Sí'],
              filled=True, rounded=True, fontsize=8, proportion=True, impurity=False, ax=ax)
    ax.set_title(f'Árbol de decisión — {c} (primeros 3 niveles; profundidad total = {best.named_steps["modelo"].get_depth()})')
    plt.show()
    print(export_text(best.named_steps['modelo'], feature_names=nombres, max_depth=3, show_weights=True))

**Variables más relevantes.**

_[COMPLETAR tras revisar los resultados.]_